# Kaggle Complete Notebook: EagleVision Phase 1 Geometric Depth Improvement

This notebook runs the complete Phase 1 workflow for the EagleVision project on Kaggle.

It does all of the following in one place:

1. clone and install the repository
2. inspect the Kaggle dataset
3. normalize the dataset into the ScanNet-style layout expected by the repo
4. download a Depth Anything V2 checkpoint
5. generate Kaggle-local configs
6. run baseline evaluation
7. train the Phase 1 adaptation model
8. run adapted evaluation
9. compare baseline vs adapted metrics
10. export artifacts and a zip bundle

Confirmed dataset path for this notebook:

- `/kaggle/input/datasets/klein2111/scannet-2d/scannet_2d`


## What Improvement We Are Testing

The proposed improvement is not a new learned renderer. The proposed improvement is a geometry-first training environment for improving a monocular depth estimator.

We compare:

- **baseline**: frozen Depth Anything V2
- **adapted**: Depth Anything V2 plus a lightweight residual adapter trained with round-trip geometric supervision

The key claim is that the adapted model should become more geometrically useful while preserving reasonable raw depth quality.


In [ ]:
from pathlib import Path
import csv
import json
import os
import shutil
import subprocess
import zipfile

import yaml


In [ ]:
KAGGLE_INPUT = Path('/kaggle/input')
KAGGLE_WORKING = Path('/kaggle/working')

REPO_URL = 'https://github.com/alooboii/EagleVision.git'
REPO_DIR = KAGGLE_WORKING / 'EagleVision'

# Confirmed dataset path for this Kaggle runtime
RAW_DATASET_DIR = Path('/kaggle/input/datasets/klein2111/scannet-2d/scannet_2d')

print('KAGGLE_INPUT:', KAGGLE_INPUT)
print('KAGGLE_WORKING:', KAGGLE_WORKING)
print('RAW_DATASET_DIR:', RAW_DATASET_DIR)

if not RAW_DATASET_DIR.exists():
    raise FileNotFoundError(f'Dataset path not found: {RAW_DATASET_DIR}')


## Shell Helper

The helper below is deliberately non-fragile for Kaggle. It can capture stdout and stderr without hiding the actual failure reason when a command exits nonzero.


In [ ]:
def run(cmd, cwd=None, capture=False, check=True):
    print('$', ' '.join(cmd))
    print(f'[status] cwd={cwd if cwd else Path.cwd()}')
    print('[status] command started')

    process = subprocess.Popen(
        cmd,
        cwd=str(cwd) if cwd else None,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        bufsize=1,
    )

    output_lines = []
    if process.stdout is not None:
        for line in process.stdout:
            print(line, end='')
            output_lines.append(line)

    return_code = process.wait()
    print(f'[status] command finished with exit code {return_code}')

    completed = subprocess.CompletedProcess(
        args=cmd,
        returncode=return_code,
        stdout=''.join(output_lines),
        stderr='',
    )

    if check and completed.returncode != 0:
        raise RuntimeError(
            f"Command failed with exit code {completed.returncode}: {' '.join(cmd)}"
        )
    return completed


def is_git_repo(path: Path) -> bool:
    return (path / '.git').exists()


## Clone Or Refresh The Repository

This cell is careful about Kaggle workspaces where a leftover directory may exist without actually being a git checkout.


In [ ]:
if REPO_DIR.exists() and not is_git_repo(REPO_DIR):
    print(f'Removing non-git directory at {REPO_DIR}')
    shutil.rmtree(REPO_DIR)

if is_git_repo(REPO_DIR):
    run(['git', 'fetch', '--all'], cwd=REPO_DIR)
    run(['git', 'pull', '--ff-only'], cwd=REPO_DIR)
else:
    run(['git', 'clone', REPO_URL, str(REPO_DIR)])

run(['python', '-m', 'pip', 'install', '-q', '-e', '.[dev]'], cwd=REPO_DIR)
print('repo ready:', REPO_DIR)


In [ ]:
TRAIN_RUN_OUTPUT_DIR = REPO_DIR / 'outputs' / 'phase1_kaggle_complete'
CONFIG_DIR = REPO_DIR / 'outputs' / 'kaggle_configs'
TARGET_DATA_ROOT = REPO_DIR / 'data' / 'scannet'

for path in [TRAIN_RUN_OUTPUT_DIR, CONFIG_DIR, TARGET_DATA_ROOT]:
    path.mkdir(parents=True, exist_ok=True)

print('TRAIN_RUN_OUTPUT_DIR:', TRAIN_RUN_OUTPUT_DIR)
print('CONFIG_DIR:', CONFIG_DIR)
print('TARGET_DATA_ROOT:', TARGET_DATA_ROOT)


## Inspect The Dataset

Before making any assumptions, inspect the first part of the directory tree. The screenshot you provided indicates scene folders with `color`, `depth`, `label`, and `pose`, which is exactly what we want.


In [ ]:
preview = [str(path) for path in sorted(RAW_DATASET_DIR.iterdir())[:80]]
preview


## Discover ScanNet-Style Scenes

We expect each scene to contain:

- `color/`
- `depth/`
- `pose/`

The extra `label/` directory is ignored.


In [ ]:
def discover_scene_dirs(root: Path):
    scenes = []
    for candidate in sorted(root.iterdir()):
        if not candidate.is_dir():
            continue
        if (candidate / 'color').exists() and (candidate / 'depth').exists() and (candidate / 'pose').exists():
            scenes.append(candidate)
    return scenes

scene_dirs = discover_scene_dirs(RAW_DATASET_DIR)
print('num scenes found:', len(scene_dirs))
print('first scenes:', [p.name for p in scene_dirs[:10]])

if not scene_dirs:
    raise RuntimeError('No valid scene directories found.')


## Normalize Into `data/scannet`

The repository expects data to live under `data/scannet`. We symlink scenes where possible and copy only if necessary.


In [ ]:
def materialize_scene(scene_dir: Path, target_root: Path):
    dst = target_root / scene_dir.name
    if dst.exists():
        return dst
    try:
        os.symlink(scene_dir, dst, target_is_directory=True)
    except OSError:
        shutil.copytree(scene_dir, dst)
    return dst

materialized = [materialize_scene(scene_dir, TARGET_DATA_ROOT) for scene_dir in scene_dirs]
scene_ids = sorted([p.name for p in TARGET_DATA_ROOT.iterdir() if p.is_dir()])

print('normalized scene count:', len(scene_ids))
print('example normalized scenes:', scene_ids[:10])


In [ ]:
first_scene = TARGET_DATA_ROOT / scene_ids[0]
print('scene:', first_scene.name)
print('color sample:', [p.name for p in sorted((first_scene / 'color').glob('*'))[:5]])
print('depth sample:', [p.name for p in sorted((first_scene / 'depth').glob('*'))[:5]])
print('pose sample:', [p.name for p in sorted((first_scene / 'pose').glob('*'))[:5]])


## Train / Validation Split

We create a simple scene-based split for the Kaggle run.


In [ ]:
MAX_SCENES = 12
if len(scene_ids) > MAX_SCENES:
    scene_ids = scene_ids[:MAX_SCENES]
split_index = max(1, int(0.8 * len(scene_ids)))
train_scenes = scene_ids[:split_index]
val_scenes = scene_ids[split_index:] if split_index < len(scene_ids) else scene_ids[:1]
print('num train scenes:', len(train_scenes))
print('num val scenes:', len(val_scenes))
print('train sample:', train_scenes[:10])
print('val sample:', val_scenes[:10])


## Download The Baseline Checkpoint

For Kaggle, we use the `vits` metric-depth model with the `hypersim` indoor profile.


In [ ]:
print('[status] downloading Depth Anything V2 checkpoint')
run(
    [
        'python', '-m', 'baseline.depth_anything_v2', 'download',
        '--mode', 'metric',
        '--profile', 'hypersim',
        '--encoder', 'vits',
    ],
    cwd=REPO_DIR,
)


In [ ]:
checkpoint_dir = REPO_DIR / 'baseline' / 'depth_anything_v2' / 'checkpoints'
checkpoint_files = list(checkpoint_dir.glob('*.pth'))
print('checkpoint dir:', checkpoint_dir)
print('checkpoint files:', checkpoint_files)

if not checkpoint_files:
    raise RuntimeError('No checkpoint file found after download.')


## Build Kaggle-Local Train And Eval Configs

We generate notebook-local YAML files with real scene lists populated from this Kaggle session.


In [ ]:
device_name = 'cuda' if shutil.which('nvidia-smi') else 'cpu'
intrinsics = [
    [577.8706, 0.0, 159.5],
    [0.0, 577.8706, 119.5],
    [0.0, 0.0, 1.0],
]
train_config = {
    'seed': 7,
    'device': device_name,
    'output_dir': str(TRAIN_RUN_OUTPUT_DIR),
    'data': {
        'root': str(TARGET_DATA_ROOT),
        'image_size': [160, 224],
        'intrinsics': intrinsics,
        'pairing': {
            'min_translation_m': 0.02,
            'max_translation_m': 0.30,
            'min_rotation_deg': 1.0,
            'max_rotation_deg': 10.0,
            'max_index_gap': 12,
            'frame_stride': 3,
            'max_frames_per_scene': 150,
            'max_pairs_per_scene': 200,
        },
        'splits': {
            'train': {'scenes': train_scenes},
            'val': {'scenes': val_scenes},
        },
    },
    'model': {
        'depth': {
            'mode': 'metric',
            'encoder': 'vits',
            'profile': 'hypersim',
            'checkpoint_path': None,
            'freeze_backbone': True,
            'adapter_hidden_channels': 32,
        }
    },
    'train': {
        'batch_size': 1,
        'epochs': 1,
        'max_steps_per_epoch': 300,
        'lr': 1e-4,
        'weight_decay': 1e-4,
        'log_interval': 10,
        'vis_interval': 50,
        'checkpoint_interval': 100,
    },
    'eval': {'batch_size': 1, 'max_batches': 200},
    'losses': {
        'weights': {
            'target_rgb': 1.0,
            'cycle_rgb': 1.0,
            'cycle_depth': 0.5,
            'target_depth': 0.25,
        }
    },
}
eval_config = {
    'device': device_name,
    'data': {
        'root': str(TARGET_DATA_ROOT),
        'image_size': [160, 224],
        'intrinsics': intrinsics,
        'pairing': {
            'min_translation_m': 0.02,
            'max_translation_m': 0.30,
            'min_rotation_deg': 1.0,
            'max_rotation_deg': 10.0,
            'max_index_gap': 12,
            'frame_stride': 3,
            'max_frames_per_scene': 150,
            'max_pairs_per_scene': 200,
        },
        'splits': {
            'val': {'scenes': val_scenes},
        },
    },
    'model': {
        'depth': {
            'mode': 'metric',
            'encoder': 'vits',
            'profile': 'hypersim',
            'checkpoint_path': None,
            'freeze_backbone': True,
            'adapter_hidden_channels': 32,
        }
    },
    'eval': {'batch_size': 1, 'max_batches': 200},
    'losses': {
        'weights': {
            'target_rgb': 1.0,
            'cycle_rgb': 1.0,
            'cycle_depth': 0.5,
            'target_depth': 0.25,
        }
    },
}
train_config_path = CONFIG_DIR / 'train_phase1_complete.yaml'
eval_config_path = CONFIG_DIR / 'eval_phase1_complete.yaml'
with train_config_path.open('w', encoding='utf-8') as handle:
    yaml.safe_dump(train_config, handle, sort_keys=False)
with eval_config_path.open('w', encoding='utf-8') as handle:
    yaml.safe_dump(eval_config, handle, sort_keys=False)
print('train config:', train_config_path)
print('eval config:', eval_config_path)
print('device:', device_name)


## Metric Parser

The evaluation CLI prints metrics to stdout. This helper turns that text into a dictionary for comparison and export.


In [ ]:
def parse_metric_output(stdout_text: str):
    metrics = {}
    for line in stdout_text.splitlines():
        if ': ' not in line:
            continue
        key, value = line.split(': ', 1)
        try:
            metrics[key.strip()] = float(value.strip())
        except ValueError:
            continue
    return metrics


## Baseline Evaluation

This measures the frozen baseline before any adaptation.


In [ ]:
print('[status] starting baseline evaluation')
baseline_eval = run(
    ['python', '-m', 'eaglevision.cli.eval', '--config', str(eval_config_path), '--baseline-only'],
    cwd=REPO_DIR,
    capture=True,
    check=False,
)

print('baseline return code:', baseline_eval.returncode)
if baseline_eval.returncode != 0:
    raise RuntimeError('Baseline evaluation failed. Read streamed output above.')


In [ ]:
baseline_metrics = parse_metric_output(baseline_eval.stdout)
baseline_metrics_path = TRAIN_RUN_OUTPUT_DIR / 'baseline_metrics.yaml'

with baseline_metrics_path.open('w', encoding='utf-8') as handle:
    yaml.safe_dump(baseline_metrics, handle, sort_keys=False)

baseline_metrics


## Train The Phase 1 Adaptation Model

This is the proposed improvement step: geometry-first round-trip supervision over a mostly frozen depth backbone with a lightweight adaptation head.


In [ ]:
print('[status] starting Phase 1 training run')
train_run = run(
    ['python', '-m', 'eaglevision.cli.train', '--config', str(train_config_path)],
    cwd=REPO_DIR,
    capture=True,
    check=False,
)

print('train return code:', train_run.returncode)
if train_run.returncode != 0:
    raise RuntimeError('Training failed. Read streamed output above.')


In [ ]:
checkpoint_dir = TRAIN_RUN_OUTPUT_DIR / 'checkpoints'
checkpoints = sorted(checkpoint_dir.glob('*.pt'))
print('checkpoints:', checkpoints)

if not checkpoints:
    raise RuntimeError(f'No checkpoints found in {checkpoint_dir}')

latest_checkpoint = checkpoints[-1]
latest_checkpoint


## Adapted Evaluation

This measures the trained adapted model using the latest checkpoint from the training run.


In [ ]:
print('[status] starting adapted evaluation')
adapted_eval = run(
    ['python', '-m', 'eaglevision.cli.eval', '--config', str(eval_config_path), '--checkpoint', str(latest_checkpoint)],
    cwd=REPO_DIR,
    capture=True,
    check=False,
)

print('adapted return code:', adapted_eval.returncode)
if adapted_eval.returncode != 0:
    raise RuntimeError('Adapted evaluation failed. Read streamed output above.')


In [ ]:
adapted_metrics = parse_metric_output(adapted_eval.stdout)
adapted_metrics_path = TRAIN_RUN_OUTPUT_DIR / 'adapted_metrics.yaml'

with adapted_metrics_path.open('w', encoding='utf-8') as handle:
    yaml.safe_dump(adapted_metrics, handle, sort_keys=False)

adapted_metrics


## Compare Baseline And Adapted Metrics

This is the central proof step for the proposed improvement.

Interpretation:

- lower is better for `target_rgb_l1`, `cycle_rgb_l1`, `reprojection_depth`, `depth_l1`, `rmse`, `abs_rel`
- higher is better for `psnr` and `ssim`


In [ ]:
metric_names = sorted(set(baseline_metrics) | set(adapted_metrics))
comparison_rows = []

for metric_name in metric_names:
    baseline_value = baseline_metrics.get(metric_name)
    adapted_value = adapted_metrics.get(metric_name)
    delta = None
    if baseline_value is not None and adapted_value is not None:
        delta = adapted_value - baseline_value
    comparison_rows.append(
        {
            'metric': metric_name,
            'baseline': baseline_value,
            'adapted': adapted_value,
            'delta': delta,
        }
    )

comparison_rows


In [ ]:
comparison_json_path = TRAIN_RUN_OUTPUT_DIR / 'comparison_metrics.json'
comparison_csv_path = TRAIN_RUN_OUTPUT_DIR / 'comparison_metrics.csv'

with comparison_json_path.open('w', encoding='utf-8') as handle:
    json.dump(comparison_rows, handle, indent=2)

with comparison_csv_path.open('w', encoding='utf-8', newline='') as handle:
    writer = csv.DictWriter(handle, fieldnames=['metric', 'baseline', 'adapted', 'delta'])
    writer.writeheader()
    writer.writerows(comparison_rows)

print(comparison_json_path)
print(comparison_csv_path)


## Optional Inference Artifact

This produces one depth-preview artifact from the normalized dataset.


In [ ]:
first_scene = TARGET_DATA_ROOT / scene_ids[0]
sample_image = next((first_scene / 'color').glob('*.jpg'))
infer_output = TRAIN_RUN_OUTPUT_DIR / 'sample_infer_depth.png'

infer_run = run(
    [
        'python', '-m', 'eaglevision.cli.infer_depth',
        '--input', str(sample_image),
        '--output', str(infer_output),
        '--mode', 'metric',
        '--encoder', 'vits',
        '--profile', 'hypersim',
    ],
    cwd=REPO_DIR,
    capture=True,
    check=False,
)

print('infer return code:', infer_run.returncode)
infer_output


## Produced Artifacts

List the main artifacts generated by this run.


In [ ]:
artifact_paths = sorted(str(path.relative_to(REPO_DIR)) for path in TRAIN_RUN_OUTPUT_DIR.rglob('*'))
artifact_paths[:200]


## Package Outputs For Export

This creates a zip file containing the run artifacts, metric summaries, and checkpoint outputs for later reuse.


In [ ]:
archive_base = REPO_DIR / 'outputs' / 'phase1_kaggle_complete_export'
archive_path = shutil.make_archive(str(archive_base), 'zip', root_dir=TRAIN_RUN_OUTPUT_DIR)
archive_path


## Final Summary

This notebook demonstrates the full proposed Phase 1 workflow:

- baseline depth evaluation
- geometry-first adaptation training
- adapted evaluation
- direct baseline vs adapted comparison

If the adapted model improves the geometry-facing metrics while maintaining acceptable depth quality, that supports the central project hypothesis.
